imports

In [11]:
import os
import numpy as np
import SimpleITK as sitk
import nibabel as nib
from tqdm import tqdm
import random

paths

In [4]:
DATA_ROOT = "../data/Han-Seg/Han-Seg/set_1"
OUTPUT_ROOT = "../data/processed"

IMAGES_OUT = os.path.join(OUTPUT_ROOT, "imagesTr")
LABELS_OUT = os.path.join(OUTPUT_ROOT, "labelsTr")

os.makedirs(IMAGES_OUT, exist_ok=True)
os.makedirs(LABELS_OUT, exist_ok=True)


organ mapping

In [5]:
ORGAN_MAP = {
    "Bone_Mandible": 1,
    "Brainstem": 2,
    "SpinalCord": 3,
    "Parotid_L": 4,
    "Parotid_R": 5,
    "Cavity_Oral": 6
}


utility functions

In [6]:
def read_nrrd(path):
    image = sitk.ReadImage(path)
    array = sitk.GetArrayFromImage(image)  # (Z, Y, X)
    spacing = image.GetSpacing()
    return array, spacing

# Resample to Fixed Spacing - 1.5 x 1.5 x 1.5 mm
def resample_image(image, spacing, new_spacing=(1.5,1.5,1.5), is_label=False):
    sitk_image = sitk.GetImageFromArray(image)
    sitk_image.SetSpacing(spacing)

    original_size = sitk_image.GetSize()
    original_spacing = sitk_image.GetSpacing()

    new_size = [
        int(round(original_size[i] * (original_spacing[i] / new_spacing[i])))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)

    if is_label:
        resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    else:
        resampler.SetInterpolator(sitk.sitkBSpline)

    resampled = resampler.Execute(sitk_image)
    return sitk.GetArrayFromImage(resampled)


def normalize_ct(ct_array):
    ct_array = np.clip(ct_array, -200, 300)
    mean = ct_array.mean()
    std = ct_array.std()
    ct_array = (ct_array - mean) / (std + 1e-8)
    return ct_array


def save_nifti(array, save_path):
    nifti = nib.Nifti1Image(array.astype(np.float32), np.eye(4))
    nib.save(nifti, save_path)


Merge All Organ Masks Into One Label Volume

In [7]:
def create_label_volume(case_folder, case_name, ct_shape):
    label_volume = np.zeros(ct_shape, dtype=np.uint8)

    for organ_name, organ_id in ORGAN_MAP.items():
        seg_file = f"{case_name}_OAR_{organ_name}.seg.nrrd"
        seg_path = os.path.join(case_folder, seg_file)

        if os.path.exists(seg_path):
            seg_array, _ = read_nrrd(seg_path)
            label_volume[seg_array > 0] = organ_id

    return label_volume


Full Preprocessing Loop

In [9]:
cases = sorted([
    f for f in os.listdir(DATA_ROOT)
    if f.startswith("case_") and os.path.isdir(os.path.join(DATA_ROOT, f))
])
print("Found cases:", cases)

for case in tqdm(cases):
    case_path = os.path.join(DATA_ROOT, case)

    # Load CT
    ct_path = os.path.join(case_path, f"{case}_IMG_CT.nrrd")
    ct_array, spacing = read_nrrd(ct_path)

    # Create merged label
    label_array = create_label_volume(case_path, case, ct_array.shape)

    # Resample
    ct_resampled = resample_image(ct_array, spacing, is_label=False)
    label_resampled = resample_image(label_array, spacing, is_label=True)

    # Normalize CT
    ct_resampled = normalize_ct(ct_resampled)

    # Save
    save_nifti(ct_resampled, os.path.join(IMAGES_OUT, f"{case}.nii.gz"))
    save_nifti(label_resampled, os.path.join(LABELS_OUT, f"{case}.nii.gz"))

print("Preprocessing Complete")


Found cases: ['case_01', 'case_02', 'case_03', 'case_04', 'case_05', 'case_06', 'case_07', 'case_08', 'case_09', 'case_10', 'case_11', 'case_12', 'case_13', 'case_14', 'case_15', 'case_16', 'case_17', 'case_18', 'case_19', 'case_20', 'case_21', 'case_22', 'case_23', 'case_24', 'case_25', 'case_26', 'case_27', 'case_28', 'case_29', 'case_30', 'case_31', 'case_32', 'case_33', 'case_34', 'case_35', 'case_36', 'case_37', 'case_38', 'case_39', 'case_40', 'case_41', 'case_42']


100%|██████████| 42/42 [18:02<00:00, 25.77s/it]

Preprocessing Complete ✅


In [10]:
label = nib.load("../data/processed/labelsTr/case_01.nii.gz").get_fdata()
print("Unique labels:", np.unique(label))

Unique labels: [0. 1. 2. 3. 4. 5. 6.]


In [12]:
PROCESSED_IMAGES = "../data/processed/imagesTr"

cases = sorted([
    f.replace(".nii.gz", "")
    for f in os.listdir(PROCESSED_IMAGES)
    if f.endswith(".nii.gz")
])

print("Total cases:", len(cases))

Total cases: 42


In [13]:
random.seed(42)
random.shuffle(cases)

In [15]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

folds = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(cases)):
    train_cases = [cases[i] for i in train_idx]
    val_cases = [cases[i] for i in val_idx]

    folds.append({
        "train": train_cases,
        "val": val_cases
    })

    print(f"\nFold {fold_idx}")
    print("Train:", len(train_cases))
    print("Val:", len(val_cases))


Fold 0
Train: 33
Val: 9

Fold 1
Train: 33
Val: 9

Fold 2
Train: 34
Val: 8

Fold 3
Train: 34
Val: 8

Fold 4
Train: 34
Val: 8


In [16]:
SPLIT_DIR = "../data/splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

for fold_idx, fold in enumerate(folds):
    fold_dir = os.path.join(SPLIT_DIR, f"fold_{fold_idx}")
    os.makedirs(fold_dir, exist_ok=True)

    with open(os.path.join(fold_dir, "train.txt"), "w") as f:
        for case in fold["train"]:
            f.write(case + "\n")

    with open(os.path.join(fold_dir, "val.txt"), "w") as f:
        for case in fold["val"]:
            f.write(case + "\n")

print("5-fold splits saved ")


5-fold splits saved 
